四种验证

1.Pydantic

In [3]:
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field, SecretStr
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY')),
    base_url="http://127.0.0.1:8889",
)

#采用pydantic会做严格校验
class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

model_with_structured = model_deepseek.with_structured_output(MovieModel)

response = model_with_structured.invoke("给出盗梦空间的信息")

rprint(response)

ValidationError: 2 validation errors for MovieModel
title
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
year
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

2. TypedDict

In [9]:
from langchain_deepseek import ChatDeepSeek
from pydantic import SecretStr
from typing_extensions import Annotated, TypedDict
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY')),
    base_url="http://127.0.0.1:8889",
)

##采用TypedDict不会做严格校验
class MovieModel(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "电影上映年份"]
    director: Annotated[str, "导演"]
    rating: Annotated[str, "电影评分，满分十分"]

model_with_structured = model_deepseek.with_structured_output(MovieModel)

response = model_with_structured.invoke("给出盗梦空间的信息")

rprint(response)

{'title1': '盗梦空间', 'year1': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}

3. JSON schema

In [10]:
from langchain_deepseek import ChatDeepSeek
from pydantic import SecretStr
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY')),
    base_url="http://127.0.0.1:8889",
)

json_schema_1 = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structured = model_deepseek.with_structured_output(json_schema_1, method="json_schema")

response = model_with_structured.invoke("给出盗梦空间的信息")

rprint(response)

{'title1': '盗梦空间', 'year1': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}

4. @dataclass

In [14]:
from dataclasses import dataclass
from langchain_deepseek import ChatDeepSeek
from pydantic import SecretStr
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY')),
    base_url="http://127.0.0.1:8889",
)

@dataclass
class Movie():
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

# 设置结构化输出
model_with_structured = model_deepseek.with_structured_output(Movie, include_raw=True)

response = model_with_structured.invoke("给我介绍下电影《星际穿越》")

rprint(response)

{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 1,
                'prompt_tokens': 1,
                'total_tokens': 2,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_provider': 'deepseek',
            'model_name': 'any',
            'system_fingerprint': None,
            'id': 'chatcmpl-test',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--01a0ba0b-381b-73f1-b5f9-222864d9efc7-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title1': '盗梦空间', 'year1': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3},
                'id': 'call_1',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 1,
            'output_tokens': 1,
            'total_tokens': 2,
            'input_token_details': {},
            'output_token_details': {}
        }
    ),
    'parsed': {'title1': '盗梦空间', 'year1': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3},
    'parsing_error': None
}